# 02 — SCD Patterns and Audit Columns

Demonstrates Slowly Changing Dimension (SCD) patterns using PySpark:

- SHA-256 hash-based change detection (business key, Type I, Type II)
- Adding standard audit columns to dimension and fact DataFrames
- Detecting new vs. changed vs. unchanged records
- Applying Type I (in-place) and Type II (versioned) update logic
- Caching DataFrames for repeated use

## Tasty Bytes Consulting — Client Dimension SCD

This notebook walks through SCD change detection for `dim_clients` —
the master client dimension for Tasty Bytes Consulting. Incoming
CRM batches are compared against the warehouse snapshot using SHA-256
fingerprints so that only genuinely changed records trigger upserts.

In [ ]:
import sys
import datetime

sys.path.insert(0, '.')
from demo_data import create_spark_session, load_all, COMPANY_NAME

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, sha2, concat_ws, lit, when
from pyspark.sql.types import StringType, BinaryType, TimestampType, BooleanType

spark = create_spark_session('SCD_Patterns')
spark.sparkContext.setLogLevel('WARN')
print(f'{COMPANY_NAME} — SCD Patterns demo')
print(f'Spark version: {spark.version}')

## 1. Helper: hash a sorted set of columns

Hashing produces a compact, comparable fingerprint of a record's values.
Sorting column names ensures the hash is deterministic regardless of
DataFrame column order.

In [ ]:
def make_hash(columns: list):
    '''Return a SHA-256 Column expression over *columns* (sorted, pipe-delimited).'''
    sorted_cols = sorted(columns, key=str.lower)
    return sha2(
        concat_ws('|', *[col(c).cast(StringType()) for c in sorted_cols]),
        256,
    ).cast(BinaryType())


def add_dimension_audit_columns(
    df: DataFrame,
    business_key_cols: list,
    type_i_cols: list = None,
    created_col: str = None,
    modified_col: str = None,
) -> DataFrame:
    '''
    Append SCD audit columns to a dimension DataFrame.

    Columns added
    -------------
    __BusinessKeyHash          - fingerprint of natural/business key
    __Type1Hash                - fingerprint of in-place update columns
    __Type2Hash                - fingerprint of versioned (history) columns
    __DeletedFlag              - False for all active records
    __CreateDateTime           - pipeline run timestamp
    __DataCreationDateTime     - source record creation time (or null)
    __DataModificationDateTime - source record modification time (or null)
    '''
    type_i_cols = type_i_cols or []
    audit_cols  = {created_col, modified_col} - {None}
    data_cols   = [c for c in df.columns if not c.startswith('__')]

    type_ii_cols = list(
        set(data_cols)
        - set(business_key_cols)
        - set(type_i_cols)
        - audit_cols
    )

    result = (
        df
        .withColumn('__BusinessKeyHash', make_hash(business_key_cols))
        .withColumn('__Type1Hash',
                    make_hash(type_i_cols) if type_i_cols
                    else lit(None).cast(BinaryType()))
        .withColumn('__Type2Hash',
                    make_hash(type_ii_cols) if type_ii_cols
                    else lit(None).cast(BinaryType()))
        .withColumn('__DeletedFlag',   lit(False).cast(BooleanType()))
        .withColumn('__CreateDateTime',
                    lit(datetime.datetime.now()).cast(TimestampType()))
    )

    result = result.withColumn(
        '__DataCreationDateTime',
        col(created_col).cast(TimestampType()) if created_col and created_col in df.columns
        else lit(None).cast(TimestampType()),
    )
    result = result.withColumn(
        '__DataModificationDateTime',
        col(modified_col).cast(TimestampType()) if modified_col and modified_col in df.columns
        else lit(None).cast(TimestampType()),
    )
    return result


print('Helpers defined.')

## 2. Load the client dimension and simulate a source batch

The **source** batch contains 6 client records from the latest CRM extract.
The **target** represents the current `dim_clients` snapshot in the warehouse.

| client_id | Change type | What changed                                                 |
|-----------|-------------|--------------------------------------------------------------|
| CLT-001   | unchanged   | —                                                            |
| CLT-003   | unchanged   | —                                                            |
| CLT-005   | Type II     | `risk_profile` moderate (was aggressive), `aum_usd` updated  |
| CLT-007   | Type I      | `region` north_america (was europe) — office relocation       |
| CLT-009   | Type II     | `relationship_manager` Sarah Chen (was James Whitfield)      |
| CLT-011   | new         | Brand-new client, not yet in the warehouse                   |

In [ ]:
# Load canonical client data from the shared data module
clients_df = load_all(spark)['clients']

# Source batch - 6 clients from the Q4 2024 CRM extract
src_cols = [
    'client_id', 'client_name', 'client_type',
    'risk_profile', 'region', 'relationship_manager',
    'aum_usd', 'onboarded_date',
]

source_data = [
    # CLT-001 - unchanged
    ('CLT-001', 'Apex Pension Fund',               'institutional', 'moderate',     'north_america', 'Sarah Chen',       420_000_000.0, '2018-03-15'),
    # CLT-003 - unchanged
    ('CLT-003', 'Marcus Harrington',               'individual',    'aggressive',   'north_america', 'Sarah Chen',        12_500_000.0, '2021-11-20'),
    # CLT-005 - Type II: risk_profile aggressive->moderate, aum_usd updated
    ('CLT-005', 'Pacific Growth Corp',             'institutional', 'moderate',     'apac',          'David Lim',        325_000_000.0, '2017-02-14'),
    # CLT-007 - Type I: region europe->north_america (office relocation)
    ('CLT-007', 'Nordic Wealth Partners',          'institutional', 'moderate',     'north_america', 'Helena Brandt',    195_000_000.0, '2015-09-22'),
    # CLT-009 - Type II: relationship_manager James Whitfield->Sarah Chen
    ('CLT-009', 'Riverside Healthcare Foundation', 'institutional', 'conservative', 'north_america', 'Sarah Chen',       225_000_000.0, '2014-01-30'),
    # CLT-011 - new client, not yet in the warehouse
    ('CLT-011', 'Global Infrastructure Partners',  'institutional', 'moderate',     'north_america', 'James Whitfield',  500_000_000.0, '2024-10-01'),
]
source_df = spark.createDataFrame(source_data, src_cols)

# Target - prior warehouse state (stale for CLT-005, CLT-007, CLT-009)
target_data = [
    ('CLT-001', 'Apex Pension Fund',               'institutional', 'moderate',     'north_america', 'Sarah Chen',       420_000_000.0, '2018-03-15'),
    ('CLT-003', 'Marcus Harrington',               'individual',    'aggressive',   'north_america', 'Sarah Chen',        12_500_000.0, '2021-11-20'),
    ('CLT-005', 'Pacific Growth Corp',             'institutional', 'aggressive',   'apac',          'David Lim',        310_000_000.0, '2017-02-14'),
    ('CLT-007', 'Nordic Wealth Partners',          'institutional', 'moderate',     'europe',        'Helena Brandt',    195_000_000.0, '2015-09-22'),
    ('CLT-009', 'Riverside Healthcare Foundation', 'institutional', 'conservative', 'north_america', 'James Whitfield',  225_000_000.0, '2014-01-30'),
]
target_df = spark.createDataFrame(target_data, src_cols)

print('Source batch (6 records):')
source_df.show(truncate=False)
print('Target snapshot (5 records - CLT-011 not yet loaded):')
target_df.show(truncate=False)

## 3. Add audit / hash columns

- `region` is a **Type I** column: office relocations are corrected in-place
- `risk_profile`, `relationship_manager`, and `aum_usd` are **Type II**:
  changes are versioned to preserve investment mandate history

In [ ]:
BK_COLS = ['client_id']
T1_COLS = ['region']   # in-place update - office relocation

source_enriched = add_dimension_audit_columns(
    source_df,
    business_key_cols=BK_COLS,
    type_i_cols=T1_COLS,
    created_col='onboarded_date',
)

target_enriched = add_dimension_audit_columns(
    target_df,
    business_key_cols=BK_COLS,
    type_i_cols=T1_COLS,
    created_col='onboarded_date',
)

# Cache target - it will be joined against multiple times
target_enriched.cache()
print(f'Target cached: {target_enriched.count()} rows')

print()
print('Source with audit columns (key fields):')
source_enriched.select(
    'client_id', 'client_name', 'risk_profile', 'region',
    'relationship_manager', '__DeletedFlag', '__CreateDateTime',
).show(truncate=False)

## 4. Change detection via hash comparison

Join source against target on the business key.  Compare hash columns
to classify each source record as **new**, **type_i_changed**,
**type_ii_changed**, or **unchanged**.

In [ ]:
# Prefix target columns to avoid collision after join
tgt = target_enriched.select(
    col('client_id').alias('tgt_client_id'),
    col('__BusinessKeyHash').alias('tgt_bk_hash'),
    col('__Type1Hash').alias('tgt_t1_hash'),
    col('__Type2Hash').alias('tgt_t2_hash'),
)

joined = source_enriched.join(
    tgt,
    source_enriched['client_id'] == tgt['tgt_client_id'],
    how='left',
)

classified = joined.withColumn(
    'change_type',
    when(col('tgt_bk_hash').isNull(),
         'new')
    .when(col('__Type1Hash') != col('tgt_t1_hash'),
         'type_i_changed')
    .when(col('__Type2Hash') != col('tgt_t2_hash'),
         'type_ii_changed')
    .otherwise('unchanged'),
)

print('Change classification:')
classified.select(
    'client_id', 'client_name', 'risk_profile', 'region',
    'relationship_manager', 'change_type',
).show(truncate=False)

## 5. Apply Type I and Type II update logic

- **Type I** changes: overwrite in place — the record takes the new values
- **Type II** changes: preserve history — mark the old record as superseded
  and insert a new active version
- **New** records: insert directly

In [ ]:
base_cols = src_cols + [
    '__BusinessKeyHash', '__Type1Hash', '__Type2Hash',
    '__DeletedFlag', '__CreateDateTime',
    '__DataCreationDateTime', '__DataModificationDateTime',
]

# Records to upsert (new + changed)
upsert_df = classified.filter(
    col('change_type') != 'unchanged'
).select(*base_cols)

print(f'Records to upsert: {upsert_df.count()}')
upsert_df.select(
    'client_id', 'client_name', 'risk_profile', 'region', 'relationship_manager',
).show(truncate=False)

# Unchanged records pass through unmodified
unchanged_df = classified.filter(
    col('change_type') == 'unchanged'
).select(*base_cols)
print(f'Unchanged records: {unchanged_df.count()}')

# Simulate final merged state
merged_df = unchanged_df.union(upsert_df)
print()
print(f'Final merged dim_clients: {merged_df.count()} rows')
merged_df.select(
    'client_id', 'client_name', 'risk_profile', 'region',
    'relationship_manager', 'aum_usd',
).show(truncate=False)

## 6. Fact table audit columns

Fact tables use a simpler audit pattern: a single `__FactKeyHash`
(hash of the grain / business key) plus `__DeletedFlag` and `__CreateDateTime`.

Here the grain is a **portfolio valuation event** — each portfolio valued
once per quarter-end (`portfolio_id` + `valuation_date`).

In [ ]:
valuation_data = [
    ('VAL-001', 'PRT-001', '2024-09-30', 148_500_000.0),
    ('VAL-002', 'PRT-002', '2024-09-30', 281_200_000.0),
    ('VAL-003', 'PRT-004', '2024-09-30',  13_100_000.0),
    ('VAL-004', 'PRT-005', '2024-09-30',  97_800_000.0),
    ('VAL-005', 'PRT-007', '2024-09-30', 318_750_000.0),
    ('VAL-006', 'PRT-009', '2024-09-30', 198_400_000.0),
]
valuations_df = spark.createDataFrame(
    valuation_data,
    ['valuation_id', 'portfolio_id', 'valuation_date', 'nav_usd'],
)

fact_key_cols = ['valuation_id']
fact_enriched = (
    valuations_df
    .withColumn('__FactKeyHash',
                make_hash(fact_key_cols))
    .withColumn('__DeletedFlag',  lit(False).cast(BooleanType()))
    .withColumn('__CreateDateTime',
                lit(datetime.datetime.now()).cast(TimestampType()))
)

print('Portfolio valuation events with audit columns:')
fact_enriched.show(truncate=False)
fact_enriched.printSchema()

In [ ]:
spark.stop()